In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [21]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

# Check columns
print(sample.head())
print(sample.columns)

# Dummy prediction
sample.iloc[:, 1] = "A"

# Save submission file
sample.to_csv("/kaggle/working/submission.csv", index=False)

print("submission.csv created successfully!")

   ID Prediction
0   1      A B C
1   2      A B C
2   3      A B C
3   4      A B C
4   5      A B C
Index(['ID', 'Prediction'], dtype='object')
submission.csv created successfully!


# EDA

In [10]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [2]:
print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [3]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [14]:
print(train.info())
print(train.isnull().sum())
print(train['answer'].value_counts())
print(train.duplicated().sum())
print(train.head(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB
None
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
0
   id                                             prompt  \
0   1  pick the best possible answer what is martin h...   
1   2           what is acceleratorbased lightion fusion   
2   3  determine the correct option what is the term ...   

                                               

# Text Cleaning

In [15]:
import re
import string

def clean_text(text):
    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)

    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [16]:
text_cols = ['prompt','A','B','C','D','E']

for col in text_cols:
    train[col] = train[col].apply(clean_text)
    test[col] = test[col].apply(clean_text)

In [17]:
train[['prompt','A']].head()

,prompt,A
0,pick the best possible answer what is martin h...,martin heidegger believes that humans exist wi...
1,what is acceleratorbased lightion fusion,acceleratorbased lightion fusion is a techniqu...
2,determine the correct option what is the term ...,blueshifting
3,select the most accurate option what is martin...,martin heidegger believes that humans exist wi...
4,identify the correct statement what is the con...,simultaneity is relative meaning that two even...


# TF-IDF

## Combined Text

In [18]:
# Combine prompt and options

train["combined_text"] = (
    train["prompt"] + " " +
    train["A"] + " " +
    train["B"] + " " +
    train["C"] + " " +
    train["D"] + " " +
    train["E"]
)

test["combined_text"] = (
    test["prompt"] + " " +
    test["A"] + " " +
    test["B"] + " " +
    test["C"] + " " +
    test["D"] + " " +
    test["E"]
)

In [19]:
train["combined_text"].head()

0    pick the best possible answer what is martin h...
1    what is acceleratorbased lightion fusion accel...
2    determine the correct option what is the term ...
3    select the most accurate option what is martin...
4    identify the correct statement what is the con...
Name: combined_text, dtype: object

## TF-IDF Vectorization

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X_tfidf = tfidf.fit_transform(train["combined_text"])

In [23]:
print(X_tfidf.shape)

(2000, 2878)


# Cosine Similarity Baseline

In [29]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

options = ['A','B','C','D','E']

In [30]:
def predict_top3(row):

    prompt_vec = tfidf.transform([row['prompt']])

    scores = []

    for opt in options:

        option_vec = tfidf.transform([row[opt]])

        score = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        scores.append(score)

    ranking = np.argsort(scores)[::-1]

    top3 = [options[i] for i in ranking[:3]]

    return " ".join(top3)

In [31]:
train["prediction"] = train.apply(
    predict_top3,
    axis=1
)

In [32]:
train[["answer","prediction"]].head()

,answer,prediction
0,B,C D B
1,A,C A B
2,C,E D C
3,B,C D B
4,A,E D B


# mAP@3

In [34]:
def apk(actual, predicted, k=3):

    if len(predicted) > k:
        predicted = predicted[:k]

    for i, p in enumerate(predicted):
        if p == actual:
            return 1.0 / (i + 1)

    return 0.0

In [35]:
def mapk(actuals, predictions, k=3):

    scores = []

    for actual, pred in zip(actuals, predictions):

        pred_list = pred.split()

        scores.append(
            apk(actual, pred_list, k)
        )

    return np.mean(scores)

In [36]:
score = mapk(
    train["answer"],
    train["prediction"]
)

print("mAP@3 =", score)

mAP@3 = 0.23575


# Word2Vec Embeddings

In [39]:
from gensim.models import Word2Vec

import gensim
print(gensim.__version__)

4.4.0


## Tokenize Text

In [40]:
def tokenize(text):
    return str(text).split()

In [41]:
sentences = []

for col in ['prompt','A','B','C','D','E']:
    sentences.extend(
        train[col].apply(tokenize).tolist()
    )

print("Number of sentences:", len(sentences))

Number of sentences: 12000


## Training Word2Vec

In [42]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [43]:
print(w2v_model.wv.vector_size)

100


## Sentence Embeddings

In [44]:
import numpy as np

def sentence_vector(text):

    words = str(text).split()

    vectors = [
        w2v_model.wv[word]
        for word in words
        if word in w2v_model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

## Cosine Similarity using Word2Vec

In [45]:
from sklearn.metrics.pairwise import cosine_similarity

options = ['A','B','C','D','E']

def predict_w2v(row):

    prompt_vec = sentence_vector(row['prompt'])

    scores = []

    for opt in options:

        option_vec = sentence_vector(row[opt])

        score = cosine_similarity(
            [prompt_vec],
            [option_vec]
        )[0][0]

        scores.append(score)

    ranking = np.argsort(scores)[::-1]

    top3 = [options[i] for i in ranking[:3]]

    return " ".join(top3)

## Generating Predictions

In [46]:
train["w2v_prediction"] = train.apply(
    predict_w2v,
    axis=1
)

## Evaluate

In [47]:
w2v_score = mapk(
    train["answer"],
    train["w2v_prediction"]
)

print("Word2Vec mAP@3 =", w2v_score)

Word2Vec mAP@3 = 0.32941666666666664


In [48]:
print("TF-IDF:", score)
print("Word2Vec:", w2v_score)

TF-IDF: 0.23575
Word2Vec: 0.32941666666666664


In [49]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["TF-IDF", "Word2Vec"],
    "mAP@3": [score, w2v_score]
})

results

,Model,mAP@3
0,TF-IDF,0.235750
1,Word2Vec,0.329417
